In [ ]:
!pip install pythoncrc
!pip install pandas --upgrade
!pip install geemap --upgrade

In [ ]:
import ee
import geemap
import os


In [ ]:
ee.Authenticate()

True

In [ ]:
ee.Initialize(project='XXXX')

#**Study Area Polygons**

In [ ]:
pts = ee.FeatureCollection('XXXX')

In [ ]:
DL = ee.FeatureCollection('XXXX')

In [ ]:
Sula = ee.FeatureCollection('XXXX') #SULAWESI POLYGON

In [ ]:
MMdist = ee.FeatureCollection('XXXX') #MOOR MACAQUE DISTRIBUTION

In [ ]:
bira = ee.FeatureCollection('XXXX') #LOCATION WITHIN THE SEA

In [ ]:
Map = geemap.Map(basemap="Esri.WorldImagery")
Map.setCenter(120, -5, 8)
Map

In [ ]:
roi=MMdist.geometry()

In [ ]:
empty = ee.Image().byte()
SLoutline = empty.paint(
featureCollection=MMdist,
color=1,
width=3
)
Map.addLayer(SLoutline, {"color": "red"}, "Sampling Locations")

#**Buffer points**

In [ ]:
def bufferPoints(radius, bounds):

    def func_bpc(pt):
        pt = ee.Feature(pt)
        return pt.buffer(radius).bounds()

    return func_bpc

Squares

In [ ]:
buffer_250 = pts.map(bufferPoints(250, False))
buffer_500 = pts.map(bufferPoints(500, False))
buffer_2km = pts.map(bufferPoints(707.1, False))
buffer_750 = pts.map(bufferPoints(750, False))
buffer_1000 = pts.map(bufferPoints(1000, False))
buffer_1250 = pts.map(bufferPoints(1250, False))

In [ ]:
DL_50 = DL.map(bufferPoints(50, False))
DL_100 = DL.map(bufferPoints(100, False))
DL_250 = DL.map(bufferPoints(250, False))
DL_500 = DL.map(bufferPoints(500, False))
DL_707 = DL.map(bufferPoints(707.1, False))
DL_1000 = DL.map(bufferPoints(1000, False))
DL_1250 = DL.map(bufferPoints(1250, False))

In [ ]:
import pandas as pd

# Define a list of DL buffer objects
dl_buffers = [DL_50, DL_100, DL_250, DL_500, DL_707, DL_1000, DL_1250]

# Define a list of corresponding output file names
output_files = [
    '/content/drive/MyDrive/Moor macaque survey 2023/Covariates Loud Call/DL_50_area.csv',
    '/content/drive/MyDrive/Moor macaque survey 2023/Covariates Loud Call/DL_100_area.csv',
    '/content/drive/MyDrive/Moor macaque survey 2023/Covariates Loud Call/DL_250_area.csv',
    '/content/drive/MyDrive/Moor macaque survey 2023/Covariates Loud Call/DL_500_area.csv',
    '/content/drive/MyDrive/Moor macaque survey 2023/Covariates Loud Call/DL_707_area.csv',
    '/content/drive/MyDrive/Moor macaque survey 2023/Covariates Loud Call/DL_1000_area.csv',
    '/content/drive/MyDrive/Moor macaque survey 2023/Covariates Loud Call/DL_1250_area.csv',
]

# Iterate through buffers and calculate area for each feature
for i, buffer in enumerate(dl_buffers):
    # Calculate the area of each feature in the buffer
    area_buffer = buffer.map(lambda feature: feature.set({'area': feature.geometry().area(1)})) # Added maxError=1

    # Convert the FeatureCollection to a list of dictionaries
    area_list = area_buffer.getInfo()['features']

    # Extract ID and area values
    areas_data = [{'id': feature['Name'], 'area_sq_meters': feature['properties']['area']} for feature in area_list]

    # Create a Pandas DataFrame
    area_df = pd.DataFrame(areas_data)

    # Define the output CSV file path
    output_csv_path = output_files[i]

    # Download the DataFrame as a CSV file
    area_df.to_csv(output_csv_path, index=False)

    print(f"Area calculations saved to: {output_csv_path}")

#**Settings**

In [1]:
#LIMIT STATISTICS TO MM DISTRIBUTION
def func_hyu(image):
    return image.clip(MMdist)

cliptocol = func_hyu


In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#**Dynamic World**

In [ ]:
startDate = '2020-01-01';
timeIntervalYears = 4;

endDate = ee.Date(startDate).advance(timeIntervalYears, 'years')

In [ ]:
dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').map(cliptocol);

dwClassInfo = {
  0: {'name': 'water', 'color': '#419BDF'}, # added quotes around name and color
  1: {'name': 'trees', 'color': '#397D49'}, # added quotes around name and color
  2: {'name': 'grass', 'color': '#88B053'},  # added quotes around name and color
  3: {'name': 'flooded_vegetation', 'color': '#7A87C6'}, # added quotes around name and color
  4: {'name': 'crops', 'color': '#E49635'}, # added quotes around name and color
  5: {'name': 'shrub_and_scrub', 'color': '#DFC35A'}, # added quotes around name and color
  6: {'name': 'built', 'color': '#C4281B'}, # added quotes around name and color
  7: {'name': 'bare', 'color': '#A59B8F'}, # added quotes around name and color
  8: {'name': 'snow_and_ice', 'color': '#B39FE1'}, # added quotes around name and color
}

In [ ]:
def func_blj (v):
    return ee.Dictionary(v).get(prop)
    return ee.Dictionary(dict).values().map(
        lambda v: ee.Dictionary(v).get(prop)
    )
    return ee.Dictionary(dict).values().map(
func_blj)


In [ ]:
def getClassProperty(dict, prop):

    def func_ovd (v):
        return ee.Dictionary(v).get(prop)
    return ee.Dictionary(dict).values().map(
        lambda v: ee.Dictionary(v).get(prop)
    )
    return ee.Dictionary(dict).values().map(
func_ovd)


In [ ]:
probabilityBands = getClassProperty(dwClassInfo, 'name');
dwClassPalette = getClassProperty(dwClassInfo, 'color');
dwVisParams = {"min": 0, "max": 8, "palette": dwClassPalette.getInfo()};


In [ ]:
dwTimeInterval = dw.filter(ee.Filter.date(startDate, endDate))

In [ ]:
dwTimeSeries = dwTimeInterval.select(probabilityBands)

In [ ]:
meanProbability = dwTimeSeries.reduce(ee.Reducer.mean())

In [ ]:
dwClassComposite = meanProbability.toArray().arrayArgmax().arrayGet(0).rename("label")

In [ ]:
region = MMdist.geometry()

In [ ]:
geemap.ee_export_image_to_asset(
image=dwClassComposite,
description='DW',
assetId='projects/gee-extractraster/assets/DW2023',
region=region,
scale=10,
crs="EPSG:4326",
maxPixels=10000000000
)


In [ ]:
#PERCENTAGE OF EACH LAND USE PER LOCATION
geemap.zonal_stats_by_group(
    DW2023,
    buffer_2km,
    '/XXXXX', # Changed output file name
    stat_type="PERCENTAGE",
    decimal_places=2,
    scale=10,
    tile_scale=16,
    crs="EPSG:4326")

#**Tree Height**

In [ ]:
TreeHeight = ee.ImageCollection("projects/sat-io/open-datasets/GLAD/GEDI_V27").map(cliptocol)

In [ ]:
print(TreeHeight)

In [ ]:
TreeHeightVisParams = {
    "bands": ["b1"],
    "min":10,
    "max": 50,
    "palette": ['#FFFFFF', '#00ff00']}
Map.addLayer(TreeHeight, TreeHeightVisParams, name="Tree Height")

In [ ]:
TH = ee.Image(TreeHeight.mosaic())

In [ ]:
#Threshold 10m
# Remap values in TH image.
Forest = TH.where(TH.lte(9), 0).where(TH.gt(9), 1)

In [ ]:
#pERCENTAGE OF FOREST COVER
geemap.zonal_stats_by_group(
    Forest,
    buffer_2km,
    'XXXX',
    stat_type="PERCENTAGE",
    decimal_places=2,
    scale=27.83,
    crs="EPSG:4326")

In [ ]:
#MAXIMUM TREE HEIGHT
geemap.zonal_stats(
    TH,
    buffer_2km,
    'XXXX',
    stat_type="MAXIMUM",
    scale=30)

#**Forest**

In [ ]:
ForestTypeSulawesi = ee.Image("XXXX") # Corrected asset path

In [ ]:
# Create a mask for pixels with values 5 and 6.
mask = ForestTypeSulawesi.neq(0)

# Apply the mask to the original image.  This will set the pixels with values 5 and 6 to 0.
ForestClass = ForestTypeSulawesi.updateMask(mask)

# Now 'new_image' contains the modified raster data with pixels 5 and 6 discarded.

# You can then export or further process this image as needed.
# For example, to export it as a GeoTIFF:

# geemap.ee_export_image(new_image, filename='new_forest_type.tif', scale=30, region=roi, file_per_band=False)


In [ ]:
mask = Forest10.reduce(ee.Reducer.anyNonZero())

In [ ]:
Forest = ForestTypeSulawesi.updateMask(mask).clip(MMdist)

In [ ]:
# Filter the image to keep only pixels with values 1, 3, or 4.
ForestClass = Forest.updateMask(
    Forest.eq(1).Or(Forest.eq(3)).Or(Forest.eq(4))
)


In [ ]:
geemap.zonal_stats_by_group(
    ForestClass,
    buffer_2km,
    'XXXX',
    stat_type="SUM",
    decimal_places=2,
    scale=30,
    crs="EPSG:4326")

Extract values as raster for predicting and mapping occupancy

In [ ]:
reduced = Forest.reduceRegions(
    collection= buffer_2km,
    reducer=ee.Reducer.max(),
    scale=30
)

In [ ]:
raster = reduced.reduceToImage(
    properties=['max'],  # Property containing the mean values
    reducer=ee.Reducer.first()
)

In [ ]:
# prompt: esport raster to drive

geemap.ee_export_image_to_drive(
    raster,
    description='TreeHeight_2km_raster',  # Name of the export task
    folder='earthengine/Rasters',  # Folder in your Google Drive
    scale=30,  # Resolution of the exported image
    region=buffer_2km.geometry(), # Export bounds based on the buffer geometry
    fileFormat='GeoTIFF', # Output file format
    maxPixels=10000000000
)

#**NDVI**

In [ ]:
# Import the Landsat 8 TOA image collection.
l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1').map(cliptocol)\
.filterDate('2023-03-17', '2023-03-21')


In [ ]:
image = ee.Algorithms.Landsat.simpleComposite(l8)

In [ ]:
# Compute the Normalized Difference Vegetation Index (NDVI).
nir = image.select('B5')
red = image.select('B4')
ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

In [ ]:
ndvi = image.normalizedDifference(['B5', 'B4']).rename('NDVI')

In [ ]:
# Display the result.

ndviParams = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
Map.addLayer(ndvi, ndviParams, 'NDVI image')

In [ ]:
geemap.zonal_stats(
        ndvi,
        buffer_2km,
        "XXXX",
        stat_type="MEAN",
        decimal_places=2,
        scale=30,
        tile_scale=16,
        crs="EPSG:4326"
    )

#**Human Footprint Index**

In [ ]:
HFI2019 = ee.Image('XXXX')

In [ ]:
# Reproject HFI2019 to the CRS of buffer_2km
HFI2019_reprojected = HFI2019.reproject(
    crs="EPSG:4326",
    scale=HFI2019.projection().nominalScale() # Keep the original resolution
).clip(MMdist)


# Print the CRS of the reprojected HFI2019 for verification
print('CRS of HFI2019_reprojected:', HFI2019_reprojected.projection().crs().getInfo())

# Add the reprojected image to the map (optional)
# You might need to define visualization parameters based on the range of HFI values
#Map.addLayer(HFI2019_reprojected, {}, 'HFI 2019 Reprojected to buffer_2km CRS')


CRS of HFI2019_reprojected: EPSG:4326


In [ ]:
geemap.zonal_stats(
    HFI2019_reprojected,
    buffer_2km,
    'XXXX',
    stat_type="MEAN",
    scale=30,
    crs="EPSG:4326",
    decimal_places=2
    )

Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/Moor macaque survey 2023/Covariates Occupancy Models/HFI2020_2km.csv


Extract values as raster

In [ ]:
reduced = HFI2019.reduceRegions(
    collection= buffer_2km,
    reducer=ee.Reducer.mean(),
    scale=30
)

In [ ]:
raster = reduced.reduceToImage(
    properties=['mean'],  # Property containing the mean values
    reducer=ee.Reducer.first()
)

In [ ]:
# prompt: esport raster to drive

geemap.ee_export_image_to_drive(
    raster,
    description='HumanFootprintIndex_2km_raster',  # Name of the export task
    folder='XXXX',  # Folder in your Google Drive
    scale=30,  # Resolution of the exported image
    region=buffer_2km.geometry(), # Export bounds based on the buffer geometry
    fileFormat='GeoTIFF', # Output file format
    maxPixels=10000000000
)

#**Human Population**

In [ ]:
image2020 = ee.Image('JRC/GHSL/P2023A/GHS_POP/2020').clip(MMdist)

In [ ]:
image2020 = image2020.updateMask(image2020.gt(0))

In [ ]:
geemap.zonal_stats(
        image2020,
        buffer,
        "XXXX",
        stat_type="SUM",
        decimal_places=2,
        scale=100,
        tile_scale=16,
        crs="EPSG:4326"
    )

#**Fire**

##2022

In [ ]:
dataset2022 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2022-01-01', '2022-12-31'))
fires2022 = dataset2022.select('T21').max().clip(MMdist)

##2021

In [ ]:
dataset2021 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2021-01-01', '2021-12-31'))
fires2021 = dataset2021.select('T21').max().clip(MMdist)

##2020

In [ ]:
dataset2020 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2020-01-01', '2020-12-31'))
fires2020 = dataset2020.select('T21').max().clip(MMdist)

##2019

In [ ]:
dataset2019 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2019-01-01', '2019-12-31'))
fires2019 = dataset2019.select('T21').max().clip(MMdist)

##2018

In [ ]:
dataset2018 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2018-01-01', '2018-12-31'))
fires2018 = dataset2018.select('T21').max().clip(MMdist)

##2017

In [ ]:
dataset2017 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2017-01-01', '2017-12-31'))
fires2017 = dataset2017.select('T21').max().clip(MMdist)

##2016

In [ ]:
dataset2016 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2016-01-01', '2016-12-31'))
fires2016 = dataset2016.select('T21').max().clip(MMdist)

##2015

In [ ]:
dataset2015 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2015-01-01', '2015-12-31'))
fires2015 = dataset2015.select('T21').max().clip(MMdist)

##2014

In [ ]:
dataset2014 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2014-01-01', '2014-12-31'))
fires2014 = dataset2014.select('T21').max().clip(MMdist)

##2013

In [ ]:
dataset2013 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2013-01-01', '2013-12-31'))
fires2013 = dataset2013.select('T21').max().clip(MMdist)

##2012

In [ ]:
dataset2012 = ee.ImageCollection('FIRMS').filter(ee.Filter.date('2012-01-01', '2012-12-31'))
fires2012 = dataset2012.select('T21').max().clip(MMdist)

##Analysis

In [ ]:
buffers = [buffer_2km]

# Define a list of fire datasets
fire_datasets = [fires2012, fires2013, fires2014, fires2015, fires2016,
                 fires2017, fires2018, fires2019, fires2020, fires2021,
                 fires2022]

# Define a list of corresponding output file name prefixes
output_file_prefixes = [
    'XXXX/Fire2012',
    'XXXX/Fire2013',
    'XXXX/Fire2014',
    'XXXX/Fire2015',
    'XXXX/Fire2016',
    'XXXX/Fire2017',
    'XXXX/Fire2018',
    'XXXX/Fire2019',
    'XXXX/Fire2020',
    'XXXX/Fire2021',
    'XXXX/Fire2022',
]


# Iterate through buffers
for i, buffer in enumerate(buffers):
    # Iterate through fire datasets
    for j, fire_dataset in enumerate(fire_datasets):
      output_filename = f'{output_file_prefixes[j]}_{i+250}.csv' # Assumes buffers are 250, 500 etc.
      geemap.zonal_stats(
          fire_dataset,
          buffer,
          output_filename,
          stat_type="SUM",
          scale=1000
      )


#**Elevation and Slope**

In [ ]:
# Import the MERIT global elevation dataset.
elev = ee.Image('MERIT/DEM/v1_0_3');

In [ ]:
# Calculate slope from the DEM.
slope = ee.Terrain.slope(elev);

In [ ]:
geemap.zonal_stats(
    slope,
    DL_250,
    'XXXX/Slope.csv',
    stat_type="MEAN",
    decimal_places=2,
    scale=92.77,
    tile_scale=16,
    crs="EPSG:4326")

#**Temperature**

###Survey 1 (March-April)

In [ ]:
dataset = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').map(cliptocol).filter(ee.Filter.date('2023-03-14', '2023-04-20'));
temperature = dataset.select('tmmx').mean().multiply(0.1)

In [ ]:
geemap.zonal_stats(
        temperature,
        buffer,
        "XXXX/Survey 1 (March-April)/MaxTa2km",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 2 (April-June)

In [ ]:
dataset = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').map(cliptocol).filter(ee.Filter.date('2023-04-25', '2023-06-02'));
temperature = dataset.select('tmmx').mean().multiply(0.1)

In [ ]:
geemap.zonal_stats(
        temperature,
        buffer,
        "XXXX/Survey 2 (April-June)/MaxTa2km",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 3 (June-July)

In [ ]:
dataset = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').map(cliptocol).filter(ee.Filter.date('2023-06-06', '2023-07-17'));
temperature = dataset.select('tmmx').mean().multiply(0.1)

In [ ]:
geemap.zonal_stats(
        temperature,
        buffer,
        "XXXX/Survey 3 (June-July)/MaxTa2km",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 4 (July-September)

In [ ]:
dataset = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').map(cliptocol).filter(ee.Filter.date('2023-07-28', '2023-08-06'));
temperature = dataset.select('tmmx').mean().multiply(0.1)

In [ ]:
geemap.zonal_stats(
        temperature,
        buffer,
        "XXXX/Survey 4 (July-September)/MaxTa2km",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 5 (August-September)

In [ ]:
dataset = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').map(cliptocol).filter(ee.Filter.date('2023-08-14', '2023-09-30'));
temperature = dataset.select('tmmx').mean().multiply(0.1)

In [ ]:
geemap.zonal_stats(
        temperature,
        buffer,
        "XXXX/Survey 5 (August-September)/MaxTa2km",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 6 (September-October)

In [ ]:
dataset = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').map(cliptocol).filter(ee.Filter.date('2023-09-06', '2023-10-24'));
temperature = dataset.select('tmmx').mean().multiply(0.1)

In [ ]:
geemap.zonal_stats(
        temperature,
        buffer,
        "XXXX/Survey 6 (September-October)/MaxTa2km",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

#**Rainfall**

###Survey 1 (March-April)

In [ ]:
dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').map(cliptocol).filter(ee.Filter.date('2023-03-14', '2023-04-20'));
temperature = dataset.select('precipitation').mean()

In [ ]:
geemap.zonal_stats(
        precipitation,
        buffer,
        "XXXX/Survey 1 (March-April)/Pr2km.csv",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 2 (April-June)

In [ ]:
dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').map(cliptocol).filter(ee.Filter.date('2023-04-25', '2023-06-02'));
temperature = dataset.select('precipitation').mean()

In [ ]:
geemap.zonal_stats(
        precipitation,
        buffer,
        "XXXX/Survey 2 (April-June)/Pr2km.csv",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 3 (June-July)

In [ ]:
dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').map(cliptocol).filter(ee.Filter.date('2023-06-06', '2023-07-17'));
temperature = dataset.select('precipitation').mean()

In [ ]:
geemap.zonal_stats(
        precipitation,
        buffer,
        "XXXX/Survey 3 (June-July)/Pr2km.csv",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 4 (July-September)

In [ ]:
dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').map(cliptocol).filter(ee.Filter.date('2023-07-28', '2023-08-06'));
temperature = dataset.select('precipitation').mean()

In [ ]:
geemap.zonal_stats(
        precipitation,
        buffer,
        "XXXX/Survey 4 (July-September)/Pr2km.csv",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 5 (August-September)

In [ ]:
dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').map(cliptocol).filter(ee.Filter.date('2023-08-14', '2023-09-30'));
temperature = dataset.select('precipitation').mean()

In [ ]:
geemap.zonal_stats(
        precipitation,
        buffer,
        "XXXX/Survey 5 (August-September)/Pr2km.csv",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

###Survey 6 (September-October)

In [ ]:
dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').map(cliptocol).filter(ee.Filter.date('2023-09-06', '2023-10-24'));
temperature = dataset.select('precipitation').mean()

In [ ]:
geemap.zonal_stats(
        precipitation,
        buffer,
        "XXXX/Survey 6 (September-October)/Pr2km.csv",
        stat_type="MEAN",
        decimal_places=2,
        scale=1000,
        crs="EPSG:4326"
    )

# **FUTURE PROJECTION**

In [ ]:
ssp_images = {
    'SSP1_RCP26_2050': ee.Image('XXXX/SSP1_RCP26_2050'),
    'SSP1_RCP26_2075': ee.Image('XXXX/SSP1_RCP26_2075'),
    'SSP1_RCP26_2100': ee.Image('XXXX/SSP1_RCP26_2100'),
    'SSP2_RCP45_2050': ee.Image('XXXX/SSP2_RCP45_2050'),
    'SSP2_RCP45_2075': ee.Image('XXXX/SSP2_RCP45_2075'),
    'SSP2_RCP45_2100': ee.Image('XXXX/SSP2_RCP45_2100'),
    'SSP4_RCP34_2050': ee.Image('XXXX/SSP4_RCP34_2050'),
    'SSP4_RCP34_2075': ee.Image('XXXX/SSP4_RCP34_2075'),
    'SSP4_RCP34_2100': ee.Image('XXXX/SSP4_RCP34_2100')}

for ssp_name, ssp_image in ssp_images.items():
    # Filter SSP image
    filtered_ssp = ssp_image.select('b1').updateMask(
        ssp_image.select('b1').eq(2)
        .Or(ssp_image.select('b1').eq(3))
        .Or(ssp_image.select('b1').eq(4))
    )

    # Mask TH with the mask from the filtered SSP
    TH_SSP_masked = TH.updateMask(filtered_ssp.mask())

    # Unmask the image to replace masked values with 0
    TH_SSP_unmasked = TH_SSP_masked.unmask(0)

    # Define the output file name
    output_filename = f'XXXX/MaxTreeHeight_{ssp_name}.csv'

    # Calculate zonal statistics
    geemap.zonal_stats(
        TH_SSP_unmasked,
        buffer_2km,
        output_filename,
        stat_type="MAXIMUM",
        scale=30
    )
    print(f"Zonal statistics for {ssp_name} saved to {output_filename}")

In [ ]:
ssp_images = {
    'SSP1_RCP26_2050': ee.Image('XXXX/SSP1_RCP26_2050'),
    'SSP1_RCP26_2075': ee.Image('XXXX/SSP1_RCP26_2075'),
    'SSP1_RCP26_2100': ee.Image('XXXX/SSP1_RCP26_2100'),
    'SSP2_RCP45_2050': ee.Image('XXXX/SSP2_RCP45_2050'),
    'SSP2_RCP45_2075': ee.Image('XXXX/SSP2_RCP45_2075'),
    'SSP2_RCP45_2100': ee.Image('XXXX/SSP2_RCP45_2100'),
    'SSP4_RCP34_2050': ee.Image('XXXX/SSP4_RCP34_2050'),
    'SSP4_RCP34_2075': ee.Image('XXXX/SSP4_RCP34_2075'),
    'SSP4_RCP34_2100': ee.Image('XXXX/SSP4_RCP34_2100'),
    }

for ssp_name, ssp_image in ssp_images.items():
    # Filter SSP image
    filtered_ssp = ssp_image.select('b1').updateMask(
        ssp_image.select('b1').eq(2)
        .Or(ssp_image.select('b1').eq(3))
        .Or(ssp_image.select('b1').eq(4))
    )

    # Mask TH with the mask from the filtered SSP
    FC_SSP_masked = ForestClass.updateMask(filtered_ssp.mask())

    # Unmask the image to replace masked values with 0
    FC_SSP_unmasked = FC_SSP_masked.unmask(0)

    # Define the output file name
    output_filename = f'XXXX/ForestPercentage_{ssp_name}.csv'

    # Calculate zonal statistics
    geemap.zonal_stats_by_group(
        FC_SSP_unmasked,
        buffer_2km,
        output_filename,
        stat_type="SUM",
        scale=30
    )
    print(f"Zonal statistics for {ssp_name} saved to {output_filename}")

In [ ]:
ssp_images = {
    'SSP1_RCP26_2050': ee.Image('XXXX/SSP1_RCP26_2050'),
    'SSP1_RCP26_2075': ee.Image('XXXX/SSP1_RCP26_2075'),
    'SSP1_RCP26_2100': ee.Image('XXXX/SSP1_RCP26_2100'),
    'SSP2_RCP45_2050': ee.Image('XXXX/SSP2_RCP45_2050'),
    'SSP2_RCP45_2075': ee.Image('XXXX/SSP2_RCP45_2075'),
    'SSP2_RCP45_2100': ee.Image('XXXX/SSP2_RCP45_2100'),
    'SSP4_RCP34_2050': ee.Image('XXXX/SSP4_RCP34_2050'),
    'SSP4_RCP34_2075': ee.Image('XXXX/SSP4_RCP34_2075'),
    'SSP4_RCP34_2100': ee.Image('XXXX/SSP4_RCP34_2100')
}

for ssp_name, ssp_image in ssp_images.items():
    # Define the output file name
    output_filename = f'xxxx_{ssp_name}.csv'

    # Calculate zonal statistics
    geemap.zonal_stats_by_group(
        ssp_image,
        buffer_2km,
        output_filename,
        stat_type="PERCENTAGE",
        scale=30
    )
    print(f"Zonal statistics for {ssp_name} saved to {output_filename}")